# 🎙️ Multilingual Audio RAG - Colab Edition

**GPU-accelerated RAG for English + Hindi transcripts**

This notebook runs entirely in Google Colab with T4/Tesla GPUs.

## Features:
- ✅ Auto-detects GPU availability
- ✅ Upload transcripts via Colab file browser
- ✅ GPU-accelerated embeddings
- ✅ Cloud LLM support (OpenAI/Anthropic/HuggingFace)
- ✅ Interactive step-by-step cells
- ✅ Persistent storage via Google Drive

## Instructions:
1. Open this notebook in Google Colab
2. Go to **Runtime > Change runtime type** and select **GPU**
3. Run each cell sequentially
4. Upload your transcript files when prompted
5. Ask questions in English or Hindi!

In [ ]:
# @title Step 1: Mount Google Drive and Setup
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted")

In [ ]:
# @title Step 2: Install Dependencies
!pip install -q sentence-transformers chromadb torch transformers accelerate bitsandbytes
!pip install -q openai anthropic huggingface-hub langchain-text-splitters langdetect tiktoken
!pip install -q ipywidgets

print("✅ Dependencies installed")

In [11]:
# @title Step 2b: Prepare Project Files in Colab
import os
import sys
from pathlib import Path
import subprocess

PROJECT_DIR = Path("/content/multilingual-audio-rag-colab")
REPO_URL = "https://github.com/amollate/multilingual-audio-rag-colab.git"

os.chdir("/content")

def run(cmd, cwd=None):
    result = subprocess.run(cmd, shell=True, cwd=cwd or "/content", check=False, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Command failed: {cmd}")
        print(result.stderr)
    return result

if PROJECT_DIR.exists():
    print("📂 Repo exists, pulling latest...")
    run("git pull", cwd=str(PROJECT_DIR))
else:
    print("📥 Cloning repository into /content...")
    run(f"git clone {REPO_URL} {PROJECT_DIR}")

if PROJECT_DIR.exists():
    os.chdir(PROJECT_DIR)
    sys.path.append(str(PROJECT_DIR))
    print(f"✅ Project ready at: {PROJECT_DIR}")
    print(f"   Contents: {os.listdir(PROJECT_DIR)}")
else:
    print("❌ Git clone failed. Falling back to writing source files directly...")
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    os.chdir(PROJECT_DIR)
    sys.path.append(str(PROJECT_DIR))
    print("⚠️  Running in fallback mode. Some features may be limited.")

📂 Repo exists, pulling latest...
✅ Project ready at: /content/multilingual-audio-rag-colab
   Contents: ['tests', '.git', 'requirements-colab.txt', 'notebooks', '.gitignore', 'src', 'README.md']


In [12]:
import subprocess
from pathlib import Path

PROJECT_DIR = Path("/content/multilingual-audio-rag-colab")

# Show current commit
commit = subprocess.run("git rev-parse --short HEAD", shell=True, cwd=PROJECT_DIR, capture_output=True, text=True).stdout.strip()
print(f"Current commit: {commit}")

# Show remote URL
remote = subprocess.run("git remote get-url origin", shell=True, cwd=PROJECT_DIR, capture_output=True, text=True).stdout.strip()
print(f"Remote URL: {remote}")

# Check if logger fix is present
processor_file = PROJECT_DIR / "src/ingestion/transcript_processor.py"
if processor_file.exists():
    content = processor_file.read_text()
    if "logger = logging.getLogger(__name__)" in content:
        print("✅ Logger fix is present")
    else:
        print("❌ Logger fix is missing - old code detected")
else:
    print("❌ transcript_processor.py not found")

Current commit: 93b3a08
Remote URL: https://github.com/amollate/multilingual-audio-rag-colab.git
✅ Logger fix is present


In [ ]:
# @title Step 3: GPU and Environment Check
import torch
import sys
from pathlib import Path

# Check GPU
gpu_available = torch.cuda.is_available()
print(f"\n{'='*50}")
print(f"GPU Available: {gpu_available}")
if gpu_available:
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️  No GPU detected. Running on CPU will be slower.")
print(f"{'='*50}\n")

# Check Python version
print(f"Python Version: {sys.version}")

# Setup directories
DRIVE_PATH = "/content/drive/MyDrive/multilingual-audio-rag"
DATA_DIR = Path(DRIVE_PATH) / "data"
TRANSCRIPT_DIR = DATA_DIR / "transcripts"
PROCESSED_DIR = DATA_DIR / "processed"
VECTOR_DB_PATH = PROCESSED_DIR / "chroma_db"

for dir_path in [DATA_DIR, TRANSCRIPT_DIR, PROCESSED_DIR, VECTOR_DB_PATH]:
    dir_path.mkdir(parents=True, exist_ok=True)

print(f"\n📁 Directories created at: {DRIVE_PATH}")
print(f"   - Transcripts: {TRANSCRIPT_DIR}")
print(f"   - Vector DB: {VECTOR_DB_PATH}")

In [ ]:
# @title Step 4: Upload Transcript Files
from google.colab import files
import shutil

print("📤 Upload your transcript files (.txt or .json)")
print("   You can select multiple files at once\n")

uploaded = files.upload()

# Move uploaded files to transcript directory
for filename in uploaded.keys():
    src = f"/content/{filename}"
    dst = TRANSCRIPT_DIR / filename
    shutil.move(src, dst)
    print(f"   ✅ Moved {filename} -> {dst}")

print(f"\n📊 Total transcripts uploaded: {len(uploaded)}")

In [ ]:
# @title Step 5: Configure LLM Provider
import os
from google.colab import widgets

# Get API keys from user
print("\n🔑 LLM Provider Configuration")
print("Choose your LLM provider and enter API key when prompted\n")

# For simplicity, we'll use OpenAI as default
# You can modify this to support other providers
OPENAI_API_KEY = "" # @param {type:"string"}

if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    print("✅ OpenAI API key set")
else:
    print("⚠️  No API key provided. Using HuggingFace models (slower)")

LLM_MODEL = "gpt-4o-mini" # @param ["gpt-4o-mini", "gpt-3.5-turbo", "claude-3-5-haiku-20240620"]
print(f"\n🤖 Selected LLM: {LLM_MODEL}")

In [13]:
# @title Step 6: Initialize RAG Pipeline
import sys
sys.path.append('/content/drive/MyDrive/multilingual-audio-rag-colab')

from src.config import ColabConfig
from src.pipeline.rag_pipeline import RAGPipeline

# Create config with auto-detection
config = ColabConfig(drive_mount_path="/content/drive/MyDrive/multilingual-audio-rag")

# Override with user settings
if OPENAI_API_KEY:
    config.llm_provider = "openai"
    config.openai_api_key = OPENAI_API_KEY
    config.llm_model = LLM_MODEL
else:
    config.llm_provider = "huggingface"
    config.llm_model = "HuggingFaceH4/zephyr-7b-beta"

# Setup directories
config.setup_directories()

# Initialize pipeline
print("\n🚀 Initializing RAG Pipeline...")
pipeline = RAGPipeline(config)

# Show status
status = pipeline.get_status()
print("\n📊 System Status:")
for key, value in status.items():
    print(f"   {key}: {value}")


🚀 Initializing RAG Pipeline...
Loading embedding model: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Device: cuda
GPU: Tesla T4
GPU Memory: 14.6 GB


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded.

📊 System Status:
   gpu_available: True
   gpu_info: {'available': True, 'name': 'Tesla T4', 'memory_total': 14.56317138671875, 'memory_allocated': 1.0358667373657227}
   llm_provider: huggingface
   llm_model: HuggingFaceH4/zephyr-7b-beta
   vector_store_count: 0
   embedding_model: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
   device: cuda


In [22]:
from pathlib import Path

path = Path("/content/multilingual-audio-rag-colab/src/ingestion/transcript_processor.py")
text = path.read_text()

# Ensure logging and logger exist
if "import logging" not in text:
    text = text.replace("from datetime import datetime", "import logging\nfrom datetime import datetime")
if "logger = logging.getLogger(__name__)" not in text:
    text = text.replace("# ==================== TRANSCRIPT PROCESSOR ====================", "logger = logging.getLogger(__name__)\n\n# ==================== TRANSCRIPT PROCESSOR ====================")

path.write_text(text)
print("Patched:", path)

Patched: /content/multilingual-audio-rag-colab/src/ingestion/transcript_processor.py


In [23]:
# @title Step 7: Ingest Transcripts
print("\n📚 Starting transcript ingestion...")
print("   This will:")
print("   1. Parse all uploaded transcripts")
print("   2. Split into chunks (~500 chars each)")
print("   3. Create embeddings using GPU")
print("   4. Store in ChromaDB\n")

# Run ingestion
result = pipeline.ingest_transcripts(str(TRANSCRIPT_DIR))

print("\n✅ Ingestion Complete!")
print(f"   Status: {result['status']}")
print(f"   Transcripts processed: {result.get('transcripts_processed', 0)}")
print(f"   Chunks created: {result.get('chunks_created', 0)}")
print(f"   Total documents in DB: {result.get('total_documents', 0)}")


📚 Starting transcript ingestion...
   This will:
   1. Parse all uploaded transcripts
   2. Split into chunks (~500 chars each)
   3. Create embeddings using GPU
   4. Store in ChromaDB



NameError: name 'logger' is not defined

In [19]:
from pathlib import Path

path = Path("/content/multilingual-audio-rag-colab/src/ingestion/transcript_processor.py")
text = path.read_text()
text = text.replace(
    "from datetime import datetime",
    "import logging\nfrom datetime import datetime\n\nlogger = logging.getLogger(__name__)"
)
path.write_text(text)
print("Patched:", path)

Patched: /content/multilingual-audio-rag-colab/src/ingestion/transcript_processor.py


In [24]:
from pathlib import Path
import importlib
import src.ingestion.transcript_processor as tp

# Fix in memory
tp.logger = __import__("logging").getLogger(__name__)

# Re-run ingestion with fixed module
result = tp.TranscriptProcessor().load_all_transcripts("/content/drive/MyDrive/multilingual-audio-rag/data/transcripts")
print(f"Loaded {len(result)} transcripts")

if result:
    chunks = pipeline.chunker.chunk_transcripts(result)
    embedded = pipeline.embedder.embed_documents(chunks, batch_size=64 if pipeline.config.gpu_available else 16)
    added = pipeline.vector_store.add_documents(embedded)
    print(f"Chunks: {len(chunks)}, Vectors added: {added}, Total: {pipeline.vector_store.get_count()}")
else:
    print("No transcripts found")

Loaded 26 transcripts


Batches:   0%|          | 0/23 [00:00<?, ?it/s]

Chunks: 1432, Vectors added: 1432, Total: 1432


In [21]:
from pathlib import Path
import importlib
from src.ingestion.transcript_processor import TranscriptProcessor
import src.pipeline.rag_pipeline as pipeline_module

# Patch file on disk
path = Path("/content/multilingual-audio-rag-colab/src/ingestion/transcript_processor.py")
text = path.read_text()
text = text.replace(
    "from datetime import datetime",
    "import logging\nfrom datetime import datetime\n\nlogger = logging.getLogger(__name__)"
)
path.write_text(text)

# Reload module so patch takes effect
importlib.reload(pipeline_module)

# Run ingestion with reloaded module
result = pipeline_module.RAGPipeline.__new__(pipeline_module.RAGPipeline)
# Re-bind components from current pipeline
for k, v in pipeline.__dict__.items():
    if not k.startswith('_'):
        setattr(result, k, v)

# Re-run ingestion
print("📚 Starting transcript ingestion...")
result = pipeline.ingest_transcripts("/content/drive/MyDrive/multilingual-audio-rag/data/transcripts")
print(result)

📚 Starting transcript ingestion...


NameError: name 'logger' is not defined

In [25]:
# @title Step 8: Query Your Transcripts!
import ipywidgets as widgets
from IPython.display import display, HTML

print("\n💬 Ask questions about your transcripts!")
print("   Try questions in English or Hindi\n")

# Create input widgets
question_input = widgets.Text(
    placeholder='Enter your question here...',
    description='Question:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='80%')
)

language_dropdown = widgets.Dropdown(
    options=['en', 'hi'],
    value='en',
    description='Language:',
    style={'description_width': 'initial'}
)

ask_button = widgets.Button(
    description='Ask',
    button_style='success',
    tooltip='Submit question'
)

output_area = widgets.Output()

def on_ask_button_clicked(b):
    question = question_input.value.strip()
    if not question:
        with output_area:
            output_area.clear_output()
            print("⚠️  Please enter a question")
        return

    language = language_dropdown.value

    with output_area:
        output_area.clear_output()
        print(f"\n🔍 Question: {question}")
        print(f"🌐 Language: {language}")
        print("\n⏳ Searching and generating answer...")

        try:
            result = pipeline.query(question, language=language)

            print(f"\n📝 Answer:")
            print(f"{result['answer']}")

            if result.get('sources'):
                print(f"\n📎 Sources ({result['retrieved_chunks']} chunks):")
                for source in result['sources'][:3]:
                    print(f"   - {source}")
        except Exception as e:
            print(f"\n❌ Error: {str(e)}")

ask_button.on_click(on_ask_button_clicked)

# Display widgets
display(widgets.HBox([question_input, language_dropdown, ask_button]))
display(output_area)


💬 Ask questions about your transcripts!
   Try questions in English or Hindi



Output()

In [26]:
!pip install -q bitsandbytes accelerate transformers

In [ ]:
# @title Step 8: Query Your Transcripts! (Local LLM)
import torch
import ipywidgets as widgets
from IPython.display import display
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import warnings
warnings.filterwarnings("ignore")

print("\n💬 Loading local model for inference...")
model_name = "HuggingFaceH4/zephyr-7b-beta"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)
print("✅ Local model loaded on GPU")

def ask_question(question, language="en"):
    results = pipeline.retriever.retrieve_with_rerank(question)
    context = pipeline.retriever.get_context_string(results)
    system_prompt = get_system_prompt(language)
    prompt = get_qa_prompt(context, question)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=1024,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    answer = tokenizer.decode(
        outputs[0][inputs.shape[1]:],
        skip_special_tokens=True
    )

    sources = list(set([r.get("metadata", {}).get("source", "") for r in results]))
    return {"answer": answer, "sources": sources, "retrieved_chunks": len(results)}

print("\n💬 Ask questions about your transcripts!")
print("   Try questions in English or Hindi\n")

question_input = widgets.Text(
    placeholder='Enter your question here...',
    description='Question:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='80%')
)

language_dropdown = widgets.Dropdown(
    options=['en', 'hi'],
    value='en',
    description='Language:',
    style={'description_width': 'initial'}
)

ask_button = widgets.Button(
    description='Ask',
    button_style='success',
    tooltip='Submit question'
)

output_area = widgets.Output()

def on_ask_button_clicked(b):
    question = question_input.value.strip()
    if not question:
        with output_area:
            output_area.clear_output()
            print("⚠️ Please enter a question")
        return

    language = language_dropdown.value

    with output_area:
        output_area.clear_output()
        print(f"\n🔍 Question: {question}")
        print(f"🌐 Language: {language}")
        print("\n⏳ Searching and generating answer...")

        try:
            result = ask_question(question, language=language)

            print(f"\n📝 Answer:")
            print(result['answer'])

            if result.get('sources'):
                print(f"\n📎 Sources ({result['retrieved_chunks']} chunks):")
                for source in result['sources'][:3]:
                    print(f"   - {source}")
        except Exception as e:
            print(f"\n❌ Error: {str(e)}")

ask_button.on_click(on_ask_button_clicked)

display(widgets.HBox([question_input, language_dropdown, ask_button]))
display(output_area)


💬 Loading local model for inference...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [30]:
# @title Step 9: Explore Your Data (Optional)
# Show statistics about ingested data

import pandas as pd

# Get all documents
docs = pipeline.vector_store.list_documents(limit=1000)

if docs:
    # Create DataFrame
    df = pd.DataFrame([
        {
            "id": d["id"],
            "source": d["metadata"].get("source", "unknown"),
            "language": d["metadata"].get("language", "unknown"),
            "chunk_size": d["metadata"].get("chunk_size", 0),
            "text_preview": d["text"][:100] + "..." if len(d["text"]) > 100 else d["text"]
        }
        for d in docs
    ])

    print(f"\n📊 Total chunks: {len(df)}")
    print(f"\n📈 By Language:")
    print(df["language"].value_counts())
    print(f"\n📁 By Source:")
    print(df["source"].value_counts().head(10))

    # Display sample
    print(f"\n📄 Sample chunks:")
    display(df.head(5))
else:
    print("No documents in database")


📊 Total chunks: 1000

📈 By Language:
language
en    1000
Name: count, dtype: int64

📁 By Source:
source
/content/drive/MyDrive/multilingual-audio-rag/data/transcripts/combined03_English.txt    205
/content/drive/MyDrive/multilingual-audio-rag/data/transcripts/combined00_English.txt    187
/content/drive/MyDrive/multilingual-audio-rag/data/transcripts/combined05_English.txt    168
/content/drive/MyDrive/multilingual-audio-rag/data/transcripts/combined02_English.txt    156
/content/drive/MyDrive/multilingual-audio-rag/data/transcripts/combined01_English.txt    135
/content/drive/MyDrive/multilingual-audio-rag/data/transcripts/combined06_English.txt     89
/content/drive/MyDrive/multilingual-audio-rag/data/transcripts/combined04_English.txt     60
Name: count, dtype: int64

📄 Sample chunks:


,id,source,language,chunk_size,text_preview
0,combined00_English_0_56df20a0,/content/drive/MyDrive/multilingual-audio-rag/...,en,489,I hope you will be able to hear me clearly and...
1,combined00_English_1_aee27e55,/content/drive/MyDrive/multilingual-audio-rag/...,en,52,". Subject hi aisa hai, lekin maja aayega. Isi ..."
2,combined00_English_2_32c7d3a8,/content/drive/MyDrive/multilingual-audio-rag/...,en,469,. Isi liye... ek session maine pehle hi socha ...
3,combined00_English_3_bac2986f,/content/drive/MyDrive/multilingual-audio-rag/...,en,472,". Abhi, parso hi ek research final hui hai, ji..."
4,combined00_English_4_403ca2fb,/content/drive/MyDrive/multilingual-audio-rag/...,en,476,". Aap log khud, jab koi jo beach sides me reht..."


In [28]:
def get_system_prompt(language: str = "en") -> str:
    if language == "hi":
        return "आप एक सहायक सहायक हैं जो ऑडियो ट्रांसक्रिप्ट से दिए गए संदर्भ के आधार पर प्रश्नों का उत्तर देते हैं। संदर्भ में अंग्रेजी और/या हिंदी में सामग्री हो सकती है। उपयोगकर्ता के प्रश्न की भाषा में उत्तर दें। यदि उत्तर संदर्भ में नहीं है, तो कहें 'मुझे उस सवाल का जवाब देने के लिए पर्याप्त जानकारी नहीं है।' संक्षिप्त और सटीक रहें। जानकारी न बनाएं।"
    return "You are a helpful assistant that answers questions based on the provided context from audio transcripts. The context may contain content in English and/or Hindi. Answer in the same language as the user's question. If the answer is not in the context, say 'I don't have enough information to answer that question.' Be concise and accurate. Do not make up information."

def get_qa_prompt(context: str, question: str) -> str:
    return f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer based on the context above. If the context is not sufficient, say 'I don't have enough information to answer that question.' Be concise and accurate. Do not make up information."

## 🎯 Quick Tips

1. **GPU Detection**: The notebook auto-detects your GPU. If no GPU is found, it falls back to CPU (slower).
2. **Persistence**: All data is saved to Google Drive, so it persists across sessions.
3. **Model Selection**: Choose the fastest model you have API access to for best experience.
4. **Transcript Format**: Supports `.txt` (line-numbered or plain) and `.json` formats.
5. **Languages**: Works with English, Hindi, and mixed content.

## 🔧 Troubleshooting

- **Out of Memory**: Reduce `chunk_size` in config or use smaller embedding model
- **Slow Responses**: Use GPT-4o-mini or Claude Haiku for fastest responses
- **Upload Issues**: Make sure files are .txt or .json format, UTF-8 encoded

## 📝 Notes

- First run will take longer as it downloads models
- Subsequent runs will be faster due to caching
- Vector DB is persisted in Google Drive
- You can stop the runtime when not in use